# Votação Majoritária - 3 Execuções

Este notebook combina **3 execuções** do classificador em um único arquivo final.

Para cada `pr_id`, comparamos o campo `owasp_category` das 3 execuções e escolhemos a categoria que apareceu em **pelo menos 2** delas (maioria). Quando há uma categoria vencedora, mantemos também o `nature` e o `summary` de uma das execuções que votaram nessa categoria.

Caso as 3 execuções discordem totalmente (sem maioria), o PR é registrado como conflito e usamos a primeira execução como fallback.

In [ ]:
from pathlib import Path
from collections import Counter
import json

# Caminhos das 3 execucoes a serem combinadas.
# Ajuste os nomes conforme os arquivos das suas 3 execucoes.
EXECUTION_FILES = [
    Path("../data/django_gemini_3.1-flash-lite_0_clean.json"),
    Path("../data/django_gemini_3.1-flash-lite_1_clean.json"),
    Path("../data/django_gemini_3.1-flash-lite_2_clean.json"),
]

OUTPUT_FILE = Path("../data/django_gemini_3.1-flash-lite_majority.json")

In [ ]:
def load_execution(path):
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    # Indexa por pr_id para facilitar a comparacao entre execucoes.
    return {item["pr_id"]: item for item in data}


executions = [load_execution(path) for path in EXECUTION_FILES]

for path, execution in zip(EXECUTION_FILES, executions):
    print(f"{path.name}: {len(execution)} PRs")

In [ ]:
def majority_vote(executions):
    # Considera apenas os pr_ids presentes em todas as execucoes.
    common_ids = set(executions[0])
    for execution in executions[1:]:
        common_ids &= set(execution)

    final_data = []
    conflicts = []

    for pr_id in common_ids:
        items = [execution[pr_id] for execution in executions]
        categories = [item["owasp_category"] for item in items]
        category, votes = Counter(categories).most_common(1)[0]

        if votes >= 2:
            # Mantem o registro completo de uma execucao que votou na categoria vencedora.
            chosen = next(item for item in items if item["owasp_category"] == category)
        else:
            # Sem maioria (3 categorias distintas): usa a primeira execucao como fallback.
            chosen = items[0]
            conflicts.append({"pr_id": pr_id, "categories": categories})

        final_data.append({
            "pr_id": pr_id,
            "owasp_category": chosen["owasp_category"],
            "nature": chosen["nature"],
            "summary": chosen["summary"],
        })

    return final_data, conflicts


final_data, conflicts = majority_vote(executions)
print(f"{len(final_data)} PRs no arquivo final.")
print(f"{len(conflicts)} PRs sem maioria (3 categorias distintas).")

In [ ]:
# Inspeciona os PRs em que nao houve maioria.
for conflict in conflicts[:20]:
    print(conflict["pr_id"], "->", conflict["categories"])

In [ ]:
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(final_data, f, indent=4, ensure_ascii=False)

print(f"Arquivo final salvo em: {OUTPUT_FILE}")

In [ ]:
# Estatisticas do resultado final.
count_none = sum(1 for item in final_data if item["owasp_category"] == "NONE")
count_issues = len(final_data) - count_none
print(f"{len(final_data)} PRs no total.")
print(f"{count_none} PRs com resposta vazia (NONE).")
print(f"{count_issues} PRs com issues identificadas.")

print("\nDistribuicao por owasp_category:")
for category, count in Counter(item["owasp_category"] for item in final_data).most_common():
    print(f"  {category}: {count}")